# 支線實驗：whole model 對 pool 的排序，和 element model 一致嗎？

這本 notebook 是**獨立的支線分析**，不改動設計流程。所有分析用的函數都建在這裡，
不往 `automated_promoter_library_design.py` 加東西；它只用該模組本來就有的 API
（`DesignConfig` / `load_core_model` / `ElementModelBundle` / `build_scored_pools` /
`pool_bin_edges`），所以就算主模組被還原也照跑。

## 要回答的問題

不是「兩根軸相不相關」（相關，Spearman ρ 大約 0.9），而是：

> **在 element-model 軸上差一個 bin 寬，足夠讓 whole model 同意這兩條序列的強弱順序嗎？**

這個量才決定設計出來的 version 會排成階梯還是擠成一團。原因是 `design_core_score`
在元件之間**嚴格可加**（conv1→conv2 之間沒有非線性，conv2 是凍結的稀疏 0/1 遮罩），
所以六格圖上每個 panel 的位移量，精確等於該元件的 core-space 貢獻差。

## 1. 設定與出處

`CHECKPOINT` 預設指向目前的 `weights_CorePromoter_clean.pt`。因為這個檔案會隨重訓被覆蓋，
下面會把它的大小、mtime 和 metadata 一起印出來，讓結果可以歸屬到特定一版模型。

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr

import automated_promoter_library_design as r
import recursive_corepromoter_design as legacy

DEVICE = torch.device("cpu")
CHECKPOINT = legacy.WEIGHTS_DIR / "weights_CorePromoter_clean.pt"
CACHE_DIR = legacy.PROJECT_ROOT / "outputs" / "energy_bin_cache"
OUT_DIR = legacy.PROJECT_ROOT / "outputs" / "energy_axis_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 隨機取樣的配對數，決定 agreement 曲線的雜訊水準。400 萬跑起來約數秒。
N_PAIRS = 4_000_000
RNG = np.random.default_rng(777)

# 和 recursive_design notebook 同一組設定；這裡只讀 pool，不做設計。
CONFIG = r.DesignConfig(
    consensus={
        "UP": ["TGGACTGATATATACAAAA", "CTAATCGGGGGCGATTAAG"],
        "m35": "TTGACA", "spacer": "TATGGCGCAAAATGGGG",
        "m10": "TATAAT", "DIS": "TTTTATTA", "ITS": "CAAAAAAAAG",
    },
    bg5="CCCTTTCGTCTTCACACAGCAGCAGTCAGGTAGGGAAGAGACC",
    bg3="GTCGACTCTAGA",
    gap_length=3,
    n_energy_bins=4,
    mutable_energy_fraction_ranges={
        "UP": (0.0, 0.7, 3), "m35": (0.0, 0.9, 4), "spacer": (0.0, 0.9, 4),
        "m10": (0.0, 0.8, 4), "DIS": (0.0, 1.0, 4), "ITS": (0.0, 1.0, 4),
    },
)

stat = CHECKPOINT.stat()
print("checkpoint:", CHECKPOINT.name, f"({stat.st_size} bytes, mtime_ns={stat.st_mtime_ns})")
meta_path = CHECKPOINT.with_name(CHECKPOINT.stem + "_metadata.json")
if meta_path.exists():
    print("metadata: ", json.dumps(json.loads(meta_path.read_text()), indent=2))

## 2. 本地函數：把 whole model 讀成元件層級的評分器

`design_core_score` 對 one-hot 輸入是**仿射**的，所以梯度是一個常數矩陣，與取梯度時
用的序列無關 —— 這代表下面取出的 `W` 是**精確**的逐鹼基權重，不是近似或擬合。
把 `W` 在某個元件視窗上切下來，就得到那個元件的 core-space PWM。

注意 conv1 的八個視窗並不平鋪整條序列，所以元件內部可能有**盲區**（權重恰為零的位置）。
`blind_positions()` 會把它們列出來。

In [ ]:
BASE_CODE = np.full(256, -1, dtype=np.int64)
for _i, _b in enumerate("ACGT"):
    BASE_CODE[ord(_b)] = _i


def element_starts(config, spacer_len):
    """元件在組裝後 full_sequence 裡的起始位置（與 assemble_library 的排版一致）。"""
    up = len(config.bg5)
    m35 = up + r.ELEMENT_LENGTHS["UP"] + config.gap_length
    spacer = m35 + r.ELEMENT_LENGTHS["m35"]
    m10 = spacer + int(spacer_len)
    dis = m10 + r.ELEMENT_LENGTHS["m10"]
    return {"UP": up, "m35": m35, "spacer": spacer, "m10": m10,
            "DIS": dis, "ITS": dis + r.ELEMENT_LENGTHS["DIS"]}


def m35_offset(model):
    """conv2 是凍結的稀疏 0/1 遮罩，每個 (channel, filter) 只有一個非零位置。"""
    w = model.conv2.weight.detach().cpu().numpy()
    offsets = []
    for ch in range(w.shape[0]):
        nz = np.where(np.abs(w[ch, 2]) > 1e-6)[0]   # filter slot 2 = -35 box
        assert len(nz) == 1, f"unexpected conv2 sparsity: {nz}"
        offsets.append(int(nz[0]))
    assert len(set(offsets)) == 1, offsets
    return offsets[0]


class CoreAxis:
    """用 whole model 自己的權重，對單一元件的序列評分。"""

    def __init__(self, model, config, device, spacer_len=None):
        self.spacer_len = int(spacer_len or r.ELEMENT_LENGTHS["spacer"])
        self.channel = int(legacy.CHANNEL_BY_SPACER[self.spacer_len])
        self.starts = element_starts(config, self.spacer_len)
        self.arch_start = self.starts["m35"] - m35_offset(model)
        seq_len = self.starts["ITS"] + r.ELEMENT_LENGTHS["ITS"] + len(config.bg3)

        model.eval()
        x = torch.zeros((1, 4, seq_len), dtype=torch.float32, device=device, requires_grad=True)
        model.architecture_logits(x)[0, self.channel, self.arch_start].backward()
        grad = x.grad[0].detach().cpu().numpy()
        self.W = {e: grad[:, self.starts[e]:self.starts[e] + r.ELEMENT_LENGTHS[e]].copy()
                  for e in r.ELEMENTS}

    def blind_positions(self, element):
        m = self.W[element]
        return [int(i) for i in np.flatnonzero((m.max(0) - m.min(0)) < 1e-9)]

    def score(self, element, sequences):
        mat = self.W[element]
        n = r.ELEMENT_LENGTHS[element]
        seqs = [str(s).upper() for s in sequences]
        out = np.full(len(seqs), np.nan)
        idx = [i for i, s in enumerate(seqs) if len(s) == n]
        if not idx:
            return out
        packed = np.frombuffer("".join(seqs[i] for i in idx).encode("ascii"), dtype=np.uint8)
        codes = BASE_CODE[packed].reshape(len(idx), n)
        ok = (codes >= 0).all(axis=1)
        out[np.asarray(idx)[ok]] = mat[codes.clip(min=0), np.arange(n)].sum(axis=1)[ok]
        return out


core_model = r.load_core_model(DEVICE, checkpoint_path=CHECKPOINT)
core_axis = CoreAxis(core_model, CONFIG, DEVICE)
print(f"channel {core_axis.channel}, arch_start {core_axis.arch_start}")
for e in r.ELEMENTS:
    blind = core_axis.blind_positions(e)
    print(f"  {e:7s} 長度 {r.ELEMENT_LENGTHS[e]:2d}，盲區位置 {blind if blind else '無'}")

## 3. 讀入兩根軸

element-model 軸直接用既有的 `build_scored_pools`（會吃快取）。core 軸在這裡自己算，
所以這本 notebook 不依賴主模組有沒有 `energy_axis`。

In [ ]:
element_models = r.ElementModelBundle(DEVICE)
elem_pools = r.build_scored_pools(element_models, CONFIG, CACHE_DIR)

frames = {}
for element in r.ELEMENTS:
    pool = elem_pools[element]
    frames[element] = pd.DataFrame({
        "sequence": pool["sequence"].to_numpy(),
        "elem": pool["energy"].to_numpy(float),
        "elem_bin": pool["energy_bin"].to_numpy(),
        "core": core_axis.score(element, pool["sequence"].tolist()),
    })
    print(f"{element:7s} n = {len(pool):,}")

## 4. 每個元件的一致性摘要

`p_agree_at_1_bin_width` 是最關鍵的一欄：在目前的 bin 寬之下，隨機抓兩條相隔一個 bin 寬的
序列，whole model 同意 element model 排序的機率。這個數字接近 0.5 就等於「分箱在
core 空間裡幾乎是隨機的」。

In [ ]:
def agreement_curve(elem, core, edges, n_pairs, rng):
    """P(兩根軸對同一對序列給出相同大小關係) 對 |Δelem| 的函數（隨機配對取樣）。"""
    n = len(elem)
    i, j = rng.integers(0, n, n_pairs), rng.integers(0, n, n_pairs)
    keep = i != j
    i, j = i[keep], j[keep]
    d_elem, d_core = elem[i] - elem[j], core[i] - core[j]
    agree = np.sign(d_elem) == np.sign(d_core)
    slot = np.digitize(np.abs(d_elem), edges) - 1
    out = np.full(len(edges) - 1, np.nan)
    for b in range(len(edges) - 1):
        sel = slot == b
        if sel.sum() >= 200:
            out[b] = float(agree[sel].mean())
    return out


N_CURVE_BINS = 40
rows, curves = [], {}
for element in r.ELEMENTS:
    df = frames[element].dropna(subset=["core"])
    elem, core = df["elem"].to_numpy(), df["core"].to_numpy()
    edges, *_ = r.pool_bin_edges(elem_pools[element])
    bin_width = float(edges[1] - edges[0])

    curve_edges = np.linspace(0.0, 4.0 * bin_width, N_CURVE_BINS + 1)
    curve = agreement_curve(elem, core, curve_edges, N_PAIRS, RNG)
    centres = (curve_edges[:-1] + curve_edges[1:]) / 2
    curves[element] = (centres / bin_width, curve)
    ok = np.isfinite(curve)
    reach95 = centres[ok][curve[ok] >= 0.95]

    k = min(100, len(df))
    top_elem = set(df.nlargest(k, "elem")["sequence"])
    top_core = set(df.nlargest(k, "core")["sequence"])
    binned = df.dropna(subset=["elem_bin"])
    core_bin = np.digitize(binned["core"], np.linspace(
        binned["core"].min(), binned["core"].max(), len(edges)))

    rows.append({
        "element": element,
        "n_pooled": len(df),
        "spearman_rho": float(spearmanr(elem, core).statistic),
        "pearson_r": float(np.corrcoef(elem, core)[0, 1]),
        "top100_jaccard": len(top_elem & top_core) / len(top_elem | top_core),
        "bin_width_elem_units": bin_width,
        "p_agree_at_1_bin_width": float(np.interp(bin_width, centres[ok], curve[ok])),
        "bin_widths_for_95pct": float(reach95[0] / bin_width) if len(reach95) else np.nan,
        "core_blind_positions": len(core_axis.blind_positions(element)),
        "element_length": r.ELEMENT_LENGTHS[element],
    })

summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "energy_axis_agreement.csv", index=False)
display(summary.round(3))

## 5. 圖一：兩根軸的關係形狀

每格一個元件，虛線是目前的 bin 邊界。密度用單一色相由淺到深（sequential），
因為每格只有一個系列，不需要類別配色。

In [ ]:
INK, INK_MUTED, GRID = "#1a1a19", "#6b6a63", "#d8d7d0"

fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.6), dpi=140)
for ax, element in zip(axes.ravel(), r.ELEMENTS):
    df = frames[element].dropna(subset=["core"])
    edges, *_ = r.pool_bin_edges(elem_pools[element])
    ax.hexbin(df["elem"], df["core"], gridsize=60, cmap="Blues",
              bins="log", linewidths=0, mincnt=1, zorder=1)
    for edge in edges:
        ax.axvline(edge, color=INK_MUTED, ls="--", lw=0.8, alpha=0.7, zorder=2)
    rho = summary.loc[summary.element == element, "spearman_rho"].iloc[0]
    ax.set_title(f"{element}    rho = {rho:.3f}", fontsize=12, color=INK)
    # Plot text stays English-only, matching the rest of this repo's notebooks:
    # matplotlib's default font has no CJK glyphs and renders them as tofu boxes.
    ax.set_xlabel("element-model energy (binned on this)", fontsize=9, color=INK_MUTED)
    ax.set_ylabel("core-model contribution", fontsize=9, color=INK_MUTED)
    ax.tick_params(labelsize=8, colors=INK_MUTED)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.grid(True, color=GRID, lw=0.6, alpha=0.5, zorder=0)

fig.suptitle("Element-model axis vs whole-model contribution, per candidate pool\n"
             "dashed = current bin edges; density is log-scaled", fontsize=15, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.93))
for ext in ("png", "svg"):
    fig.savefig(OUT_DIR / f"energy_axis_scatter.{ext}", bbox_inches="tight", facecolor="white")
plt.show()

## 6. 圖二：要差多遠，whole model 才會同意？

橫軸是以**該元件自己的 bin 寬**為單位的間距，所以六格可以直接互相比較。
在 x = 1 的虛線上讀值，就是「相鄰兩個 version 的排序有多可靠」。

畫成小倍數而不是六條疊在同一張圖上：其中四個元件的曲線幾乎重合、都在 x < 1 就飽和，
疊圖時線和標籤都會互相蓋掉。每格只有一條有色線加上其他五條的灰色背景，
辨識靠的是格子位置和標題，不是顏色。

In [ ]:
ACCENT, BACKDROP = "#2a78d6", "#c9c8c1"

fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.2), dpi=140, sharex=True, sharey=True)
for ax, element in zip(axes.ravel(), r.ELEMENTS):
    for other in r.ELEMENTS:                      # 其他五條當灰色背景，方便跨格比較
        if other == element:
            continue
        xs, ys = curves[other]
        ok = np.isfinite(ys)
        ax.plot(xs[ok], ys[ok], lw=1.2, color=BACKDROP, zorder=2)

    ax.axhline(0.95, color=INK_MUTED, ls=":", lw=1.0, zorder=1)
    ax.axvline(1.0, color=INK_MUTED, ls="--", lw=1.0, zorder=1)
    xs, ys = curves[element]
    ok = np.isfinite(ys)
    ax.plot(xs[ok], ys[ok], lw=2.2, color=ACCENT, zorder=3, solid_capstyle="round")

    row = summary.loc[summary.element == element].iloc[0]
    reach = row.bin_widths_for_95pct
    reach_txt = f"95% at {reach:.2f} bw" if np.isfinite(reach) else "never reaches 95%"
    ax.set_title(f"{element}    rho {row.spearman_rho:.3f}    {reach_txt}",
                 fontsize=11, color=INK)
    ax.tick_params(labelsize=8, colors=INK_MUTED)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.grid(True, color=GRID, lw=0.6, alpha=0.5, zorder=0)

axes[0][0].set_xlim(0, 4.0)
axes[0][0].set_ylim(0.5, 1.01)
fig.suptitle("How far apart must two candidates be before the whole model agrees?\n"
             "grey = the other five elements; dotted = 95%; dashed = one bin width",
             fontsize=15, color=INK)
fig.supxlabel("separation on the element-model axis, in units of that element's bin width",
              fontsize=11, color=INK)
fig.supylabel("P(whole model orders the pair the same way)", fontsize=11, color=INK)
fig.tight_layout(rect=(0.012, 0.015, 1, 0.91))
for ext in ("png", "svg"):
    fig.savefig(OUT_DIR / f"energy_axis_agreement.{ext}", bbox_inches="tight", facecolor="white")
plt.show()
print("written to:", OUT_DIR)

## 怎麼讀這兩張圖

- **ρ 高但曲線在 x = 1 附近很低** → 全域排序一致，但相鄰 version 的差距被模型歧異蓋過。
  這正是「分箱等距、圖上卻擠在一起」的成因。
- **曲線要到 x = 2、3 才過 0.95** → 該元件目前的 bin 寬不夠，version 之間需要拉更開，
  或者改用 core 軸分箱。
- **散布圖出現明顯非線性或反轉** → 兩個模型對該元件的看法有系統性分歧，
  不只是雜訊，值得單獨查那個元件的權重。

盲區位置多的元件（UP、spacer）要留意：core 軸對那些鹼基完全看不到，
所以在那個維度上的排序差異必然被記為「不一致」。